# Object-Oriented Programming in Python — The sklearn Way

## Why OOP for ML?

Every sklearn model follows the same interface:
```python
model = Ridge(alpha=1.0)          # __init__: set hyperparameters
model.fit(X_train, y_train)       # fit: learn from data
y_pred = model.predict(X_test)    # predict: apply what was learned
score = model.score(X_test, y_test) # score: evaluate
```

This consistency is not accidental — it's a carefully designed OOP interface.  
Understanding it means we can:
- Build **custom transformers** that plug into sklearn Pipelines
- Build **custom estimators** that behave like any sklearn model
- Read and understand sklearn/PyTorch/TensorFlow source code
- Design our own ML components correctly

## Who This Notebook Is For

This notebook focuses on **Python's specific OOP semantics** and  
**how they're applied in the ML ecosystem** — not "what is inheritance."

## What We Cover

| Section | Topic |
|---|---|
| 1 | Classes — Python specifics (self, __init__, dunder methods) |
| 2 | Inheritance & super() — how sklearn uses it |
| 3 | Properties & Descriptors — @property, @staticmethod, @classmethod |
| 4 | Magic Methods — making objects work with Python operators |
| 5 | Dataclasses — the modern way to write config/data objects |
| 6 | Building a Real sklearn-Compatible Transformer |
| 7 | Building a Real sklearn-Compatible Estimator |


In [9]:
import numpy as np
# import math
# from copy import deepcopy
from dataclasses import dataclass, field
from typing import  List
from abc import ABC, abstractmethod

print("Libraries loaded ✓")


Libraries loaded ✓


---
## Section 1 — Python Classes: The Specifics

### `self` Is Not Magic — It's Just the First Argument

In Python, `self` is a convention (we could name it anything, but don't).  
When we call `model.fit(X, y)`, Python translates this to `Ridge.fit(model, X, y)`.  
`self` is the instance being operated on.

### `__init__` vs `__new__`

- `__new__` creates the object (rarely overridden)
- `__init__` initializes the object — this is the constructor we use

### Instance Attributes vs Class Attributes

```python
class Ridge:
    default_alpha = 1.0            # CLASS attribute — shared across all instances

    def __init__(self, alpha=1.0):
        self.alpha = alpha         # INSTANCE attribute — unique to this instance
        self.coef_ = None          # underscore suffix = "set after fit()"
```


In [ ]:
# ── Building a class from scratch ─────────────────────────────────────────────
from typing import Optional


print("=== Python Class Fundamentals ===")
print()

class LinearModel:
    # Class attribute: shared across ALL instances
    n_instances = 0
    model_family = 'linear'

    def __init__(self, learning_rate: float = 0.01, max_iter: int = 1000):
        # Instance attributes: unique to each object
        self.learning_rate = learning_rate
        self.max_iter = max_iter

        # Convention in sklearn: attributes set by fit() end with underscore
        self.coef_: Optional[np.ndarray] = None
        self.intercept_: Optional[float] = None
        self.n_iter_: Optional[int] = None
        self.is_fitted_: bool = False

        # Track how many instances exist
        LinearModel.n_instances += 1

    def fit(self, X, y):
        # Validate
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        # Closed-form solution: w = (X^T X)^{-1} X^T y
        X_b = np.column_stack([np.ones(len(X)), X])   # add bias column
        w   = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        self.intercept_ = w[0]
        self.coef_      = w[1:]
        self.is_fitted_ = True
        return self   # return self so we can chain: model.fit(X, y).predict(X)

    def predict(self, X):
        if not self.is_fitted_:
            raise RuntimeError("Call fit() before predict()")
        X = np.array(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return X @ self.coef_ + self.intercept_

    def score(self, X, y):
        y_pred = self.predict(X)
        y = np.array(y, dtype=float)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1 - ss_res / ss_tot

    def __repr__(self):
        # repr: unambiguous representation — what we see in notebooks/debugger
        return (f"LinearModel(learning_rate={self.learning_rate}, "
                f"max_iter={self.max_iter})")

    def __str__(self):
        # str: human-readable
        status = f"fitted (coef={self.coef_})" if self.is_fitted_ else "not fitted"
        return f"LinearModel [{status}]"


# ── Use it ────────────────────────────────────────────────────────────────────
np.random.seed(42)
X = np.random.randn(50, 2)
y = 2 * X[:, 0] - 1.5 * X[:, 1] + np.random.randn(50) * 0.5

m1 = LinearModel(learning_rate=0.001)
m2 = LinearModel(learning_rate=0.01, max_iter=500)

print(f"  repr:  {repr(m1)}")
print(f"  str:   {str(m1)}")
print(f"  Instances created: {LinearModel.n_instances}")
print()

m1.fit(X, y)
print("  After fit:")
assert m1.coef_ is not None
print(f"    coef_:      {m1.coef_.round(3)}")
print(f"    intercept_: {m1.intercept_:.4f}")
print(f"    R²:         {m1.score(X, y):.4f}")
print(f"    str:   {str(m1)}")
print()

# Method chaining (fit returns self)
r2 = LinearModel().fit(X, y).score(X, y)
print(f"  Chained: LinearModel().fit(X, y).score(X, y) = {r2:.4f}")
print()

# Class vs instance attributes
print(f"  Class attribute (shared): LinearModel.model_family = '{LinearModel.model_family}'")
print(f"  Instance 1 lr: {m1.learning_rate}   Instance 2 lr: {m2.learning_rate}")


=== Python Class Fundamentals ===

  repr:  LinearModel(learning_rate=0.001, max_iter=1000)
  str:   LinearModel [not fitted]
  Instances created: 2

  After fit:
    coef_:      [ 1.937 -1.526]
    intercept_: -0.0300
    R²:         0.9483
    str:   LinearModel [fitted (coef=[ 1.93706237 -1.52571586])]

  Chained: LinearModel().fit(X, y).score(X, y) = 0.9483

  Class attribute (shared): LinearModel.model_family = 'linear'
  Instance 1 lr: 0.001   Instance 2 lr: 0.01


---
## Section 2 — Inheritance & super()

### sklearn's Class Hierarchy

```
BaseEstimator          ← provides get_params() / set_params()
    ├── RegressorMixin    ← provides score() for regressors
    ├── ClassifierMixin   ← provides score() for classifiers
    └── TransformerMixin  ← provides fit_transform()

Ridge(BaseEstimator, RegressorMixin)
Lasso(BaseEstimator, RegressorMixin)
StandardScaler(BaseEstimator, TransformerMixin)
```

By inheriting from `BaseEstimator`, our custom model gets `get_params()` and  
`set_params()` for free — which means it works with `GridSearchCV` automatically.

By inheriting from `RegressorMixin`, we get `score()` for free (uses R²).

### `super()` — Calling the Parent Class

```python
class MyRidge(BaseEstimator, RegressorMixin):
    def __init__(self, alpha=1.0):
        super().__init__()   # call BaseEstimator.__init__()
        self.alpha = alpha
```


In [11]:
# ── Inheritance hierarchy ──────────────────────────────────────────────────────
print("=== Inheritance ===")
print()

# Abstract base class — defines the interface all models must implement
class BaseRegressor(ABC):
    # Abstract methods MUST be implemented by subclasses
    @abstractmethod
    def fit(self, X, y):
        pass

    @abstractmethod
    def predict(self, X):
        pass

    # Concrete method — provided to all subclasses
    def score(self, X, y):
        # Default: R² score
        y_pred = self.predict(X)
        y = np.array(y, dtype=float)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot

    def __repr__(self):
        params = ', '.join(f'{k}={v}' for k, v in vars(self).items()
                          if not k.endswith('_'))
        return f"{type(self).__name__}({params})"


class RidgeRegressor(BaseRegressor):
    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha         # ONLY store hyperparameters in __init__

    def fit(self, X, y):
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        X_b = np.column_stack([np.ones(len(X)), X])
        # Ridge normal equation: w = (X^T X + αI)^{-1} X^T y
        n_features = X_b.shape[1]
        reg_matrix  = self.alpha * np.eye(n_features)
        reg_matrix[0, 0] = 0        # don't regularize the bias
        self.coef_      = np.linalg.solve(X_b.T @ X_b + reg_matrix, X_b.T @ y)
        self.intercept_ = self.coef_[0]
        self.coef_      = self.coef_[1:]
        return self

    def predict(self, X):
        X = np.array(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return X @ self.coef_ + self.intercept_


class ElasticNetRegressor(BaseRegressor):
    def __init__(self, alpha: float = 1.0, l1_ratio: float = 0.5, max_iter: int = 1000):
        self.alpha     = alpha
        self.l1_ratio  = l1_ratio
        self.max_iter  = max_iter

    def fit(self, X, y):
        # Simplified implementation using sklearn under the hood
        from sklearn.linear_model import ElasticNet
        self._model = ElasticNet(alpha=self.alpha, l1_ratio=self.l1_ratio,
                                  max_iter=self.max_iter)
        self._model.fit(X, y)
        self.coef_      = self._model.coef_
        self.intercept_ = self._model.intercept_
        return self

    def predict(self, X):
        return self._model.predict(X)


# ── Demonstrate ───────────────────────────────────────────────────────────────
np.random.seed(0)
X = np.random.randn(80, 3)
y = 2*X[:,0] - 1.5*X[:,1] + 0.5*X[:,2] + np.random.randn(80) * 0.3

models = [
    RidgeRegressor(alpha=0.1),
    RidgeRegressor(alpha=10.0),
    ElasticNetRegressor(alpha=0.1, l1_ratio=0.5),
]

print(f"  {'Model':>40}  {'R²':>6}")
print("  " + "-" * 52)
for model in models:
    model.fit(X, y)
    r2 = model.score(X, y)  # score() comes from BaseRegressor
    print(f"  {repr(model):>40}  {r2:.4f}")

print()
# isinstance checks respect the hierarchy
print(f"  isinstance(RidgeRegressor(), BaseRegressor): "
      f"{isinstance(RidgeRegressor(), BaseRegressor)}")
print("  MRO (Method Resolution Order) for RidgeRegressor:")
for cls in RidgeRegressor.__mro__:
    print(f"    {cls}")
print()

# ── super() in practice ────────────────────────────────────────────────────────
print("=== super() ===")

class TimedRegressor(RidgeRegressor):
    # Adds timing to fit() without rewriting it
    def __init__(self, alpha=1.0, verbose=True):
        super().__init__(alpha=alpha)   # call RidgeRegressor.__init__
        self.verbose = verbose
        self.fit_time_:Optional[float] = None

    def fit(self, X, y):
        import time
        t0 = time.perf_counter()
        super().fit(X, y)              # call RidgeRegressor.fit
        self.fit_time_ = time.perf_counter() - t0
        if self.verbose:
            print(f"  Fit completed in {self.fit_time_*1000:.3f} ms")
        return self

timed = TimedRegressor(alpha=1.0, verbose=True)
timed.fit(X, y)
print(f"  R²: {timed.score(X, y):.4f}")
assert timed.fit_time_ is not None
print(f"  Fit time stored: {timed.fit_time_*1000:.3f} ms")


=== Inheritance ===

                                     Model      R²
  ----------------------------------------------------
                 RidgeRegressor(alpha=0.1)  0.9888
                RidgeRegressor(alpha=10.0)  0.9779
  ElasticNetRegressor(alpha=0.1, l1_ratio=0.5, max_iter=1000, _model=ElasticNet(alpha=0.1))  0.9834

  isinstance(RidgeRegressor(), BaseRegressor): True
  MRO (Method Resolution Order) for RidgeRegressor:
    <class '__main__.RidgeRegressor'>
    <class '__main__.BaseRegressor'>
    <class 'abc.ABC'>
    <class 'object'>

=== super() ===
  Fit completed in 0.033 ms
  R²: 0.9887
  Fit time stored: 0.033 ms


---
## Section 3 — Properties, Static & Class Methods

### `@property` — Computed Attributes

Instead of calling `model.get_n_features()`, we write `model.n_features_in_` —
it looks like an attribute but runs code under the hood.

```python
@property
def n_features_in_(self):
    if not self.is_fitted_:
        raise AttributeError("Model not fitted yet")
    return len(self.coef_)
```

### `@staticmethod` vs `@classmethod`

```python
class Model:
    @staticmethod
    def validate_alpha(alpha):
        # No access to instance (self) or class (cls)
        # Just a utility function namespaced under Model
        return alpha > 0

    @classmethod
    def from_config(cls, config: dict):
        # cls = the class itself — can create new instances
        # Useful as alternative constructors
        return cls(alpha=config['alpha'])
```


In [ ]:
print("=== @property ===")
print()

class RidgeModel:
    def __init__(self, alpha=1.0):
        self._alpha    = alpha   # private by convention (single underscore)
        self.coef_     = None

    @property
    def alpha(self):
        # Getter — called when we access model.alpha
        return self._alpha

    @alpha.setter
    def alpha(self, value):
        # Setter — called when we do model.alpha = 2.0
        if value <= 0:
            raise ValueError(f"alpha must be > 0, got {value}")
        self._alpha = float(value)

    @property
    def is_fitted(self):
        # Read-only property (no setter)
        return self.coef_ is not None

    @property
    def n_features_in_(self):
        if not self.is_fitted:
            raise AttributeError("Model not fitted yet. Call fit() first.")
        assert self.coef_ is not None  # type narrowing
        return len(self.coef_)

    def fit(self, X, y):
        X = np.atleast_2d(np.array(X, dtype=float))
        y = np.array(y, dtype=float)
        X_b = np.column_stack([np.ones(len(X)), X])
        reg = self._alpha * np.eye(X_b.shape[1])
        reg[0,0] = 0
        w = np.linalg.solve(X_b.T @ X_b + reg, X_b.T @ y)
        self.coef_ = w[1:]
        return self

    @staticmethod
    def validate_input(X, y):
        # Utility: doesn't need self or cls — pure function
        X = np.array(X)
        y = np.array(y)
        if len(X) != len(y):
            raise ValueError(f"X and y length mismatch: {len(X)} vs {len(y)}")
        if np.any(np.isnan(X)) or np.any(np.isnan(y)):
            raise ValueError("Input contains NaN values")
        return True

    @classmethod
    def from_config(cls, config: dict):
        # Alternative constructor
        return cls(alpha=config.get('alpha', 1.0))

    @classmethod
    def from_json(cls, json_str: str):
        import json
        config = json.loads(json_str)
        return cls.from_config(config)


# Demo
model = RidgeModel(alpha=1.0)
print(f"  model.alpha:     {model.alpha}")
print(f"  model.is_fitted: {model.is_fitted}")

# Property setter with validation
try:
    model.alpha = -1.0   # should raise
except ValueError as e:
    print(f"  Validation: {e}")

model.alpha = 2.0
print(f"  After model.alpha = 2.0: {model.alpha}")
print()

# After fitting
np.random.seed(0)
X = np.random.randn(30, 4)
y = X @ np.array([1, -2, 0.5, 1.5]) + 0.1 * np.random.randn(30)
model.fit(X, y)
print(f"  model.is_fitted:     {model.is_fitted}")
print(f"  model.n_features_in_: {model.n_features_in_}")
print()

# staticmethod
print(f"  Validation (static):  {RidgeModel.validate_input(X, y)}")
print()

# classmethod: alternative constructors
config_dict = {'alpha': 0.5, 'normalize': False}
m2 = RidgeModel.from_config(config_dict)
print(f"  from_config: alpha = {m2.alpha}")

m3 = RidgeModel.from_json('{"alpha": 0.01}')
print(f"  from_json:   alpha = {m3.alpha}")


=== @property ===

  model.alpha:     1.0
  model.is_fitted: False
  Validation: alpha must be > 0, got -1.0
  After model.alpha = 2.0: 2.0

  model.is_fitted:     True
  model.n_features_in_: 4

  Validation (static):  True

  from_config: alpha = 0.5
  from_json:   alpha = 0.01


---
## Section 4 — Magic (Dunder) Methods

Python uses **dunder methods** (double underscore) to make objects work  
with built-in operations like `+`, `len()`, `[]`, `str()`, `repr()`.

| Dunder | Triggered by |
|---|---|
| `__repr__` | `repr(obj)`, debugger, notebooks |
| `__str__` | `str(obj)`, `print(obj)` |
| `__len__` | `len(obj)` |
| `__getitem__` | `obj[key]` |
| `__setitem__` | `obj[key] = value` |
| `__contains__` | `x in obj` |
| `__iter__` | `for x in obj` |
| `__add__` | `obj + other` |
| `__eq__` | `obj == other` |
| `__call__` | `obj(args)` — makes instance callable! |


In [13]:
print("=== Dunder Methods ===")
print()

class Dataset:
    # A dataset class that behaves like a built-in sequence
    def __init__(self, X, y, feature_names=None):
        self.X = np.array(X, dtype=float)
        self.y = np.array(y, dtype=float)
        self.feature_names = feature_names or [f'feat_{i}' for i in range(self.X.shape[1])]

    def __len__(self):
        # len(dataset) → number of samples
        return len(self.X)

    def __getitem__(self, idx):
        # dataset[5] → (X[5], y[5])
        # dataset[0:10] → (X[0:10], y[0:10])
        return self.X[idx], self.y[idx]

    def __contains__(self, label):
        # label in dataset → True if label is in y
        return label in self.y

    def __iter__(self):
        # for x, y in dataset → yields (X[i], y[i]) pairs
        for i in range(len(self)):
            yield self.X[i], self.y[i]

    def __repr__(self):
        return (f"Dataset(n_samples={len(self)}, "
                f"n_features={self.X.shape[1]}, "
                f"features={self.feature_names})")

    def __add__(self, other):
        # dataset1 + dataset2 → concatenated dataset
        new_X = np.vstack([self.X, other.X])
        new_y = np.concatenate([self.y, other.y])
        return Dataset(new_X, new_y, self.feature_names)


class Scaler:
    # __call__ makes the scaler callable like a function
    def __init__(self):
        self.mean_ = None
        self.std_  = None

    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0) + 1e-8
        return self

    def __call__(self, X):
        # scaler(X) instead of scaler.transform(X)
        if self.mean_ is None:
            raise RuntimeError("Call fit() first")
        return (X - self.mean_) / self.std_

    def __repr__(self):
        fitted = f"fitted (mean={self.mean_.round(2)})" if self.mean_ is not None else "not fitted"
        return f"Scaler({fitted})"


# ── Demo ──────────────────────────────────────────────────────────────────────
np.random.seed(1)
X1 = np.random.randn(30, 3)
y1 = (X1[:,0] > 0).astype(float)
X2 = np.random.randn(10, 3)
y2 = (X2[:,0] > 0).astype(float)

ds1 = Dataset(X1, y1, ['age', 'income', 'score'])
ds2 = Dataset(X2, y2, ['age', 'income', 'score'])

print(f"  repr:     {repr(ds1)}")
print(f"  len:      {len(ds1)} samples")
print(f"  ds1[0]:   X={ds1[0][0].round(3)}  y={ds1[0][1]}")
print(f"  ds1[0:3]: {len(ds1[0:3][0])} samples")
print(f"  1.0 in ds1: {1.0 in ds1}")
print()

# Iteration
for i, (x, label) in enumerate(ds1):
    if i >= 3:
        break

# Concatenation
ds_combined = ds1 + ds2
print(f"  ds1 + ds2: {repr(ds_combined)}")
print()

# Callable scaler
scaler = Scaler().fit(X1)
X_scaled = scaler(X1)  # calling the instance like a function!
print(f"  scaler repr: {repr(scaler)}")
print(f"  scaler(X1) mean: {X_scaled.mean(axis=0).round(4)}  ← should be ~0")
print(f"  scaler(X1) std:  {X_scaled.std(axis=0).round(4)}   ← should be ~1")


=== Dunder Methods ===

  repr:     Dataset(n_samples=30, n_features=3, features=['age', 'income', 'score'])
  len:      30 samples
  ds1[0]:   X=[ 1.624 -0.612 -0.528]  y=1.0
  ds1[0:3]: 3 samples
  1.0 in ds1: True

  ds1 + ds2: Dataset(n_samples=40, n_features=3, features=['age', 'income', 'score'])

  scaler repr: Scaler(fitted (mean=[-0.12  0.19  0.11]))
  scaler(X1) mean: [ 0. -0. -0.]  ← should be ~0
  scaler(X1) std:  [1. 1. 1.]   ← should be ~1


---
## Section 5 — Dataclasses — Modern Config & Data Objects

Python 3.7 introduced `@dataclass` — a decorator that auto-generates  
`__init__`, `__repr__`, `__eq__`, and more from field definitions.

Perfect for:
- **Hyperparameter configs** (replace massive `__init__` boilerplate)
- **Experiment tracking** (immutable records of what was run)
- **Data records** (feature vectors with named fields)

```python
@dataclass
class TrainingConfig:
    learning_rate: float = 0.001
    epochs: int = 100
    batch_size: int = 32
    optimizer: str = 'adam'
    # __init__, __repr__, __eq__ auto-generated
```


In [14]:
from dataclasses import asdict
import json

print("=== Dataclasses ===")
print()

@dataclass
class RidgeConfig:
    # Training hyperparameters — with defaults
    alpha:          float = 1.0
    fit_intercept:  bool  = True
    max_iter:       int   = 1000
    tol:            float = 1e-4
    solver:         str   = 'auto'

    def __post_init__(self):
        # Validation after __init__ runs
        if self.alpha <= 0:
            raise ValueError(f"alpha must be > 0, got {self.alpha}")
        if self.max_iter < 1:
            raise ValueError(f"max_iter must be >= 1, got {self.max_iter}")


@dataclass
class ExperimentResult:
    experiment_id: str
    model_name:    str
    config:        RidgeConfig
    train_r2:      float
    val_r2:        float
    val_rmsle:     float
    n_features:    int
    n_samples:     int
    tags:          List[str] = field(default_factory=list)  # mutable default!

    @property
    def overfit_gap(self) -> float:
        return self.train_r2 - self.val_r2

    @property
    def is_good(self) -> bool:
        return self.val_r2 > 0.85 and self.overfit_gap < 0.05


# ── Usage ─────────────────────────────────────────────────────────────────────
config = RidgeConfig(alpha=0.1, max_iter=500)
print(f"  Config: {config}")
print(f"  alpha:  {config.alpha}")
print()

# Invalid config
try:
    bad_config = RidgeConfig(alpha=-1.0)
except ValueError as e:
    print(f"  Validation: {e}")
print()

# Experiment result
result = ExperimentResult(
    experiment_id = 'exp_042',
    model_name    = 'Ridge',
    config        = config,
    train_r2      = 0.923,
    val_r2        = 0.901,
    val_rmsle     = 0.128,
    n_features    = 215,
    n_samples     = 1168,
    tags          = ['house_prices', 'baseline'],
)

print(f"  Result: {result.experiment_id}")
print(f"  Overfit gap: {result.overfit_gap:.3f}")
print(f"  Is good:     {result.is_good}")
print()

# Convert to dict / JSON for logging
result_dict = asdict(result)
print(f"  As dict keys: {list(result_dict.keys())}")

# Serialize to JSON (useful for experiment tracking)
# Note: config is nested, asdict handles it recursively
json_str = json.dumps({
    'experiment_id': result.experiment_id,
    'val_r2': result.val_r2,
    'config': asdict(result.config),
}, indent=2)
print(f"  As JSON:\n{json_str}")


=== Dataclasses ===

  Config: RidgeConfig(alpha=0.1, fit_intercept=True, max_iter=500, tol=0.0001, solver='auto')
  alpha:  0.1

  Validation: alpha must be > 0, got -1.0

  Result: exp_042
  Overfit gap: 0.022
  Is good:     True

  As dict keys: ['experiment_id', 'model_name', 'config', 'train_r2', 'val_r2', 'val_rmsle', 'n_features', 'n_samples', 'tags']
  As JSON:
{
  "experiment_id": "exp_042",
  "val_r2": 0.901,
  "config": {
    "alpha": 0.1,
    "fit_intercept": true,
    "max_iter": 500,
    "tol": 0.0001,
    "solver": "auto"
  }
}


---
## Section 6 — Building a Real sklearn-Compatible Transformer

This is the practical payoff. Build a custom transformer that:
- Works inside a sklearn `Pipeline`
- Works with `GridSearchCV`
- Has the exact same interface as `StandardScaler`, `PolynomialFeatures`, etc.

**The interface:**
```python
transformer.fit(X, y=None)          → self
transformer.transform(X)            → X_transformed
transformer.fit_transform(X, y=None) → X_transformed  (from TransformerMixin)
```

Inherit from `BaseEstimator` + `TransformerMixin` — we get  
`fit_transform()` and `get_params()` / `set_params()` for free.


In [15]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
import numpy as np

print("=== Custom sklearn Transformer ===")
print()

class WinsorizeTransformer(BaseEstimator, TransformerMixin):
    # Clips extreme values at the given percentile bounds
    # Useful for handling outliers before modeling

    def __init__(self, lower_pct: float = 5.0, upper_pct: float = 95.0):
        # IMPORTANT: parameter names in __init__ MUST match attribute names
        # BaseEstimator.get_params() uses inspect to read __init__ signature
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct

    def fit(self, X, y=None):
        # Compute bounds from training data
        X = np.array(X, dtype=float)
        self.lower_bounds_ = np.percentile(X, self.lower_pct, axis=0)
        self.upper_bounds_ = np.percentile(X, self.upper_pct, axis=0)
        self.n_features_in_ = X.shape[1]
        return self   # always return self from fit()

    def transform(self, X):
        # Apply the clipping (computed from training data only)
        X = np.array(X, dtype=float).copy()
        for j in range(X.shape[1]):
            X[:, j] = np.clip(X[:, j], self.lower_bounds_[j], self.upper_bounds_[j])
        return X


class FeatureInteractionTransformer(BaseEstimator, TransformerMixin):
    # Adds pairwise multiplication interactions between features
    # e.g. [a, b, c] → [a, b, c, a*b, a*c, b*c]

    def __init__(self, include_original: bool = True):
        self.include_original = include_original

    def fit(self, X, y=None):
        X = np.array(X)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = np.array(X, dtype=float)
        n = X.shape[1]
        interactions = [X[:, i] * X[:, j]
                        for i in range(n)
                        for j in range(i+1, n)]
        parts = ([X] if self.include_original else []) +                 [inter.reshape(-1, 1) for inter in interactions]
        return np.hstack(parts)

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = [f'x{i}' for i in range(self.n_features_in_)]
        n = len(input_features)
        names = (list(input_features) if self.include_original else [])
        names += [f'{input_features[i]}*{input_features[j]}'
                  for i in range(n) for j in range(i+1, n)]
        return names


# ── Demo: plug into a Pipeline ────────────────────────────────────────────────
np.random.seed(42)
n = 200
X_demo = np.random.randn(n, 4)
# Add some outliers
X_demo[::20] *= 5
y_demo = (X_demo[:, 0] * X_demo[:, 1]   # interaction term is real signal
          + 2 * X_demo[:, 2]
          - X_demo[:, 3]
          + np.random.randn(n) * 0.5)

# Build the pipeline — exactly like any sklearn pipeline
pipe = Pipeline([
    ('winsorize',     WinsorizeTransformer(lower_pct=2, upper_pct=98)),
    ('interactions',  FeatureInteractionTransformer(include_original=True)),
    ('ridge',         Ridge(alpha=1.0)),
])

# Cross-validation works automatically
scores = cross_val_score(pipe, X_demo, y_demo, cv=5, scoring='r2')
print("  WinsorizeTransformer + FeatureInteractionTransformer + Ridge")
print(f"  5-fold CV R²: {scores.mean():.4f} ± {scores.std():.4f}")
print()

# get_params() works (from BaseEstimator)
print("  Pipeline params:")
for k, v in pipe.get_params().items():
    if not hasattr(v, 'fit'):  # skip sub-estimator objects
        print(f"    {k}: {v}")
print()

# Feature names
pipe.fit(X_demo, y_demo)
feat_names = pipe.named_steps['interactions'].get_feature_names_out(['a', 'b', 'c', 'd'])
print(f"  Output features: {feat_names}")

# WinsorizeTransformer is fitted — it remembered the bounds from training data
print(f"  Winsorize lower bounds: {pipe.named_steps['winsorize'].lower_bounds_.round(3)}")
print(f"  Winsorize upper bounds: {pipe.named_steps['winsorize'].upper_bounds_.round(3)}")


=== Custom sklearn Transformer ===

  WinsorizeTransformer + FeatureInteractionTransformer + Ridge
  5-fold CV R²: 0.6040 ± 0.2000

  Pipeline params:
    memory: None
    steps: [('winsorize', WinsorizeTransformer(lower_pct=2, upper_pct=98)), ('interactions', FeatureInteractionTransformer()), ('ridge', Ridge())]
    transform_input: None
    verbose: False
    winsorize__lower_pct: 2
    winsorize__upper_pct: 98
    interactions__include_original: True
    ridge__alpha: 1.0
    ridge__copy_X: True
    ridge__fit_intercept: True
    ridge__max_iter: None
    ridge__positive: False
    ridge__random_state: None
    ridge__solver: auto
    ridge__tol: 0.0001

  Output features: ['a', 'b', 'c', 'd', 'a*b', 'a*c', 'a*d', 'b*c', 'b*d', 'c*d']
  Winsorize lower bounds: [-2.475 -1.952 -2.708 -2.213]
  Winsorize upper bounds: [2.062 2.993 3.082 2.74 ]


---
## Section 7 — Building a Real sklearn-Compatible Estimator

Now build a full estimator (not just a transformer) — something that does `fit()` + `predict()`.

Inherit from `BaseEstimator` + `RegressorMixin`.
we get `score()` (R²) and `get_params()` / `set_params()` for free.
GridSearchCV will work on it. Pipeline will accept it.


In [16]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.model_selection import GridSearchCV
import numpy as np

print("=== Custom sklearn Estimator ===")
print()

class HuberRegressor(BaseEstimator, RegressorMixin):
    # Huber regression: MSE for small errors, MAE for large ones
    # More robust to outliers than plain linear regression
    # Uses gradient descent (so we can see the mechanics)

    def __init__(self, delta: float = 1.35, learning_rate: float = 0.01,
                 max_iter: int = 500, tol: float = 1e-4):
        self.delta         = delta
        self.learning_rate = learning_rate
        self.max_iter      = max_iter
        self.tol           = tol

    def _huber_loss(self, residuals):
        # Huber loss: quadratic for |r| <= delta, linear for |r| > delta
        delta = self.delta
        quad  = 0.5 * residuals**2
        lin   = delta * (np.abs(residuals) - 0.5 * delta)
        return np.where(np.abs(residuals) <= delta, quad, lin)

    def _huber_gradient(self, residuals):
        # Gradient of Huber loss w.r.t. residuals
        return np.where(np.abs(residuals) <= self.delta,
                        residuals,
                        self.delta * np.sign(residuals))

    def fit(self, X, y):
        # sklearn validation utilities (checks types, NaNs, shape consistency)
        X, y = check_X_y(X, y, dtype=float)

        n_samples, n_features = X.shape
        X_b = np.column_stack([np.ones(n_samples), X])

        # Initialize weights
        w  = np.zeros(n_features + 1)
        self.loss_history_ = []

        for iteration in range(self.max_iter):
            residuals = y - X_b @ w
            grad      = -X_b.T @ self._huber_gradient(residuals) / n_samples
            w        -= self.learning_rate * grad

            loss = np.mean(self._huber_loss(residuals))
            self.loss_history_.append(loss)

            # Convergence check
            if len(self.loss_history_) > 1:
                if abs(self.loss_history_[-2] - self.loss_history_[-1]) < self.tol:
                    break

        self.coef_      = w[1:]
        self.intercept_ = w[0]
        self.n_iter_    = iteration + 1
        self.n_features_in_ = n_features
        return self

    def predict(self, X):
        check_is_fitted(self)              # raises if fit() wasn't called
        X = check_array(X, dtype=float)   # validate input
        return X @ self.coef_ + self.intercept_


# ── Demo ──────────────────────────────────────────────────────────────────────
np.random.seed(42)
n = 150
X_h = np.random.randn(n, 3)
y_h = 2*X_h[:,0] - 1.5*X_h[:,1] + np.random.randn(n) * 0.5

# Add outliers — this is where Huber beats Ridge
outlier_idx = np.random.choice(n, 15, replace=False)
y_h[outlier_idx] += np.random.choice([-10, 10], 15)

huber = HuberRegressor(delta=1.35, learning_rate=0.1, max_iter=1000)
huber.fit(X_h, y_h)
print("  HuberRegressor results:")
print(f"    Converged in {huber.n_iter_} iterations")
print(f"    Coefficients: {huber.coef_.round(3)}")
print(f"    R²:           {huber.score(X_h, y_h):.4f}")  # from RegressorMixin
print()

# Works with Pipeline
pipe_huber = Pipeline([
    ('scaler', WinsorizeTransformer()),
    ('huber',  HuberRegressor(delta=1.0)),
])
scores = cross_val_score(pipe_huber, X_h, y_h, cv=5, scoring='r2')
print(f"  In Pipeline, 5-fold CV R²: {scores.mean():.4f} ± {scores.std():.4f}")
print()

# Works with GridSearchCV!
param_grid = {
    'huber__delta':         [0.5, 1.0, 1.35, 2.0],
    'huber__learning_rate': [0.01, 0.1],
}
gs = GridSearchCV(pipe_huber, param_grid, cv=5, scoring='r2')
gs.fit(X_h, y_h)
print(f"  GridSearchCV best params: {gs.best_params_}")
print(f"  GridSearchCV best R²:     {gs.best_score_:.4f}")


=== Custom sklearn Estimator ===

  HuberRegressor results:
    Converged in 69 iterations
    Coefficients: [ 1.972 -1.572 -0.017]
    R²:           0.3979

  In Pipeline, 5-fold CV R²: 0.3614 ± 0.0853

  GridSearchCV best params: {'huber__delta': 2.0, 'huber__learning_rate': 0.1}
  GridSearchCV best R²:     0.3805


---
## Summary — OOP Patterns for ML Engineers

### The sklearn Interface Contract

```python
class MyEstimator(BaseEstimator, RegressorMixin):
    def __init__(self, hyperparam=default):
        # ONLY hyperparameters here
        # NO data processing in __init__
        self.hyperparam = hyperparam

    def fit(self, X, y):
        # Learn from data → store in attributes ending with _
        self.coef_ = ...
        return self    # always return self

    def predict(self, X):
        check_is_fitted(self)   # always validate
        return ...
```

### Key Rules

1. `__init__` only stores hyperparameters — never processes data
2. Learned attributes end with underscore: `coef_`, `mean_`, `n_iter_`
3. `fit()` returns `self` — enables method chaining and Pipeline compatibility
4. Inherit from `BaseEstimator` + `RegressorMixin` — free `get_params()`, `score()`
5. Use `@dataclass` for config objects — eliminates boilerplate
6. Use `@property` for computed attributes with validation

### Next Notebook
`python_for_ml.ipynb` — NumPy, Matplotlib, vectorization, and the ML-specific Python patterns
